# Simulation 
Here we test to see that our code works by injecting simulated sky frames with band-limited noise and see if we can recover the signal.

In [ ]:
import numpy as np

def knee_psd_timeseries(n, fs, f_knee=5.0, beta=1.0, rms=1.0, seed=None):
    """
    Real-valued time series with PSD:
      - flat (white) above f_knee
      - ~ 1/f^beta below f_knee
    beta=1 is classic 1/f; beta=2 is random-walk-ish.

    rms sets the final time-series RMS (after mean removal).
    """
    rng = np.random.default_rng(seed)
    freqs = np.fft.rfftfreq(n, d=1/fs)

    # complex white spectrum
    spec = rng.normal(size=freqs.size) + 1j * rng.normal(size=freqs.size)

    # amplitude shaping for desired PSD
    # PSD ~ |spec|^2 * shape^2  => choose shape ~ f^{-beta/2} below knee
    shape = np.ones_like(freqs)
    m = (freqs > 0) & (freqs < f_knee)
    shape[m] = (f_knee / freqs[m]) ** (beta / 2)

    # DC set to 0 (avoid huge low-f blow-up)
    shape[freqs == 0] = 0.0

    spec *= shape

    x = np.fft.irfft(spec, n=n)
    x -= np.mean(x)
    x *= (rms / (np.std(x) + 1e-12))
    return x